# NYC Taxi Trip Revenue & Operations Analysis

**Objective:** Analyse NYC taxi trip data to identify the highest revenue pickup zones, understand trip patterns, and provide data driven recommendations to improve fleet operations and revenue.

**Datasets:**
- `taxi_trips.csv` - 19,996 taxi trips including fare, tip, tolls, distance, and location data
- `zones.json` - Zone lookup table mapping location IDs to zone names and boroughs

**Tools:** Python, Pandas, Matplotlib

## 1. Setup & Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt

In [ ]:
# Set display options.
pd.set_option('display.width', 50)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 25)

In [ ]:
# Import taxi trips dataset from Google Drive.
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/taxi_trips.csv', dtype='unicode')
print('Rows:', len(df))
df.head()

In [ ]:
# Import and normalise the zones lookup from JSON.
with open('/content/drive/MyDrive/Colab Notebooks/zones.json', 'r') as f:
    zones_data = json.load(f)

zones_df = pd.json_normalize(zones_data)
print('Zones loaded:', len(zones_df))
zones_df.head()

## 2. Data Cleaning

In [ ]:
# Check for missing values across all columns.
df.isnull().sum()

In [ ]:
# Convert numeric columns from string to float.
numeric_cols = ['fare_amount', 'tip_amount', 'tolls_amount', 'total_amount', 'trip_distance', 'passenger_count']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Numeric conversion complete.')
df[numeric_cols].dtypes

In [ ]:
# Drop rows where total_amount is missing, zero, or negative.
df_clean = df.dropna(subset=['total_amount'])
df_clean = df_clean[df_clean['total_amount'] > 0]

print('Rows after cleaning total_amount:', len(df_clean))

In [ ]:
# Drop rows where trip_distance is missing or zero.
df_clean = df_clean.dropna(subset=['trip_distance'])
df_clean = df_clean[df_clean['trip_distance'] > 0]

print('Rows after cleaning trip_distance:', len(df_clean))

In [ ]:
# Parse pickup_datetime and extract date features.
df_clean['pickup_datetime'] = pd.to_datetime(df_clean['pickup_datetime'])

df_clean['hour'] = df_clean['pickup_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['pickup_datetime'].dt.day_name()
df_clean['month'] = df_clean['pickup_datetime'].dt.month

print('Datetime features extracted.')
df_clean[['pickup_datetime', 'hour', 'day_of_week', 'month']].head()

## 3. Zone Enrichment

In [ ]:
# Standardise zone_id and pickup_location_id to string for merging.
zones_df['zone_id'] = zones_df['zone_id'].astype(str)
df_clean['pickup_location_id'] = df_clean['pickup_location_id'].astype(str)

# Standardise borough to title case to fix inconsistencies in the zones data.
zones_df['borough'] = zones_df['borough'].str.title()

print('Zone IDs standardised.')
zones_df.head()

In [ ]:
# Merge taxi trips with zones on pickup location.
df_merged = pd.merge(
    df_clean,
    zones_df,
    left_on='pickup_location_id',
    right_on='zone_id',
    how='inner'
)

print('Rows after merging with zones:', len(df_merged))
df_merged[['pickup_location_id', 'zone_name', 'borough', 'total_amount']].head()

## 4. Revenue Analysis

In [ ]:
# Calculate average revenue and trip count per pickup zone.
zone_revenue = df_merged.groupby('zone_name').agg(
    total_trips=('total_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean'),
    avg_total=('total_amount', 'mean'),
    total_revenue=('total_amount', 'sum')
).round(2).reset_index()

zone_revenue = zone_revenue.sort_values('total_revenue', ascending=False)
zone_revenue.head(10)

In [ ]:
# Identify the highest revenue pickup zone.
top_zone = zone_revenue.iloc[0]
print(f"Highest revenue zone: {top_zone['zone_name']}")
print(f"Total revenue: ${top_zone['total_revenue']:,.2f}")
print(f"Average fare: ${top_zone['avg_fare']:,.2f}")
print(f"Total trips: {int(top_zone['total_trips'])}")

In [ ]:
# Calculate average revenue per borough.
borough_revenue = df_merged.groupby('borough').agg(
    total_trips=('total_amount', 'count'),
    avg_total=('total_amount', 'mean'),
    total_revenue=('total_amount', 'sum')
).round(2).reset_index()

borough_revenue = borough_revenue.sort_values('total_revenue', ascending=False)
borough_revenue

In [ ]:
# Plot total revenue by borough.
plt.figure(figsize=(8, 5))
plt.bar(borough_revenue['borough'], borough_revenue['total_revenue'], color='steelblue')
plt.title('Total Revenue by Borough')
plt.xlabel('Borough')
plt.ylabel('Total Revenue ($)')
plt.tight_layout()
plt.savefig('revenue_by_borough.png', dpi=150)
plt.show()

## 5. Trip Pattern Analysis

In [ ]:
# Find the most frequent pickup hour.
hourly_trips = df_clean['hour'].value_counts().sort_index()
peak_hour = hourly_trips.idxmax()
print(f"Peak pickup hour: {peak_hour}:00 with {hourly_trips[peak_hour]} trips")

In [ ]:
# Plot trip volume by hour of day.
plt.figure(figsize=(10, 5))
plt.bar(hourly_trips.index, hourly_trips.values, color='steelblue')
plt.title('Trip Volume by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('trips_by_hour.png', dpi=150)
plt.show()

In [ ]:
# Find the most frequent pickup and dropoff location pair.
top_route = df_clean.groupby(['pickup_location_id', 'dropoff_location_id']).size()
top_route = top_route.sort_values(ascending=False).reset_index()
top_route.columns = ['pickup_location_id', 'dropoff_location_id', 'trip_count']
print('Top 5 most frequent routes:')
top_route.head()

In [ ]:
# Calculate average tip rate as a percentage of fare.
df_clean['tip_rate'] = (df_clean['tip_amount'] / df_clean['fare_amount']) * 100
df_clean['tip_rate'] = df_clean['tip_rate'].replace([float('inf'), -float('inf')], None)
df_clean_tips = df_clean.dropna(subset=['tip_rate'])

avg_tip_rate = df_clean_tips['tip_rate'].mean()
print(f"Average tip rate: {avg_tip_rate:.2f}%")

## 6. Tolls Analysis

In [ ]:
# Drop rows where tolls_amount is zero, negative, or missing.
df_tolls = df_clean.dropna(subset=['tolls_amount'])
df_tolls = df_tolls[df_tolls['tolls_amount'] > 0]

print('Trips with tolls:', len(df_tolls))
print(f"Average toll amount: ${df_tolls['tolls_amount'].mean():.2f}")
print(f"Average total fare on toll trips: ${df_tolls['total_amount'].mean():.2f}")

In [ ]:
# Compare average total fare for toll vs non-toll trips.
df_no_tolls = df_clean[(df_clean['tolls_amount'] == 0) | (df_clean['tolls_amount'].isnull())]

print(f"Average fare (with tolls):    ${df_tolls['total_amount'].mean():.2f}")
print(f"Average fare (without tolls): ${df_no_tolls['total_amount'].mean():.2f}")

## 7. Summary & Business Recommendations

**Key Findings:**

1. **Revenue by zone**: LaGuardia Airport (zone 138) is among the highest revenue pickup zones, generating significantly above average fares per trip. Airport pickups are a disproportionately high revenue source relative to trip volume.

2. **Peak hours**: Trip volume peaks in the early evening (5pm–8pm), with a secondary peak in the morning (7am–9am). Fleet availability during these windows has the highest revenue impact.

3. **Toll trips are higher value**: Trips with tolls have a meaningfully higher average total fare, suggesting cross borough or airport routes are more lucrative per trip than short inner-city rides.

4. **Tipping behaviour**: The average tip rate across the dataset provides a baseline for understanding payment type preferences and customer satisfaction by zone.

**Recommendations:**

- **Prioritise airport and cross borough routes**: Drivers positioned near LaGuardia Airport during peak hours generate the highest revenue per trip.
- **Increase fleet availability at peak hours**: Early evening and morning commute periods drive the highest trip volume. Scheduling more drivers during these windows directly increases total revenue.
- **Investigate low revenue zones**: Zones with high trip counts but below average fares may indicate inefficient short distance routes that reduce overall fleet productivity.